In [2]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
from clusters_lattice_bigN import SpinConfiguration

def get_bibj(n = 24, delta = 1., lamb = 1., print_data = False):

    sc = SpinConfiguration(N = n, key = f'{n}A', delta = delta, lamb = lamb, lowest_eignstates=5, print_data = print_data, save_ham=False, force_ham_gen=False)
    #print(f'Ground state energy per spin: {round(sc.get_ground_state()[0]/(n), 5)}J')
    gs = sc.get_ground_state()[1]
    cluster_map = sc.cluster.cluster_map
    rotation_map = sc.rotation_map

    directions = [(-1, 0), (0, -1)]  # x and y directions
    flips = [sc.flips_change_side, sc.flips_change_up]  # x and y directions

    eval_bibj = 0.
    k = [0., 0.]
    easy_ks = [0., 2.]
    is_easy_k = easy_ks.__contains__(k[0]) and easy_ks.__contains__(k[1])

    for repr, state in tqdm(sc.representatives.items(), total = len(sc.representatives), disable = not print_data):
        config = sc.map_int_to_config_extended(repr)
        
        rot = (config * rotation_map + (1 - config) * (1 - rotation_map))

        for id, dir in enumerate(directions):
            roll_config = np.roll(config, dir, axis=(0, 1))
                            
            possible_mixing = np.mod((config + roll_config) * cluster_map, 2)

            config_int_shift = np.array(possible_mixing * flips[id], dtype=float)
            # multiplying to account for sign
            config_int_shift *= (-1.)**config

            for int_shift in config_int_shift[config_int_shift != 0].astype(int):
                repr_b, trans = sc.roll_to_repr(repr + int_shift)
                basis_index = sc.map_int_to_basis(repr_b)
                
                if basis_index < state:
                    phase = 1.
                    if not is_easy_k:
                        phase = np.mean([np.exp( -1j * np.pi * np.dot(k, t) ) for t in trans])
                    
                    element = 0.5 * phase * np.sqrt(sc.norms[repr]/sc.norms[repr_b])
                    
                    eval_bibj += 2 * np.real(np.conj(gs[basis_index]) * gs[state]) * element
                    #eval_bibj += np.conj(gs[basis_index]) * gs[state] * element

    eval_bibj /= n

    rotation_map = sc.rotation_map
    m_s = 0
    mag_int = 0
    
    for repr, amp in zip(sc.representatives.keys(), gs):
        config = sc.map_int_to_config(repr)
        rot = config * rotation_map + (1 - config) * (1 - rotation_map)

        m_s += np.square(np.abs(amp)) * np.sum(rot * cluster_map) / n
        mag_int += np.square(np.abs(amp)) * np.sum(rot * np.roll(rot, (0, -1), axis = (0, 1)) * cluster_map) / n
        mag_int += np.square(np.abs(amp)) * np.sum(rot * np.roll(rot, (-1, 0), axis = (0, 1)) * cluster_map) / n
    
    return eval_bibj, m_s, mag_int

In [ ]:
ns = np.arange(18, 26.001, 2, dtype = int)
lamb = 1.
delta = 2.

n_bs = []
n_mss = []
n_mag_int = []

for n in ns:
    bibj, ms, mag_int = get_bibj(n, delta, lamb, print_data = True)
    n_bs.append(bibj)
    n_mss.append(ms)
    n_mag_int.append(mag_int)
    print(f'{n} done')

In [ ]:
data_matrix = np.zeros((len(ns), 4))
data_matrix[:, 0] = ns
data_matrix[:, 1] = n_bs
data_matrix[:, 2] = n_mss
data_matrix[:, 3] = n_mag_int
np.savetxt(f'size_scaling_params_{float(delta)}_{float(lamb)}.txt', data_matrix)

In [ ]:
from scipy.optimize import minimize

def fit(x, pars):
    a, b, c = pars
    return a + b * x**(7/2) + c * x**(9/2)

def cost(pars):
    return np.sum(np.square(fit(1/ns, pars) + np.array(n_bs)))

x0 = [.1668, 40.0, 500.0]
fun = minimize(cost, x0)
print(fun.x)

x = np.linspace(0, 0.066, 51)
plt.plot(x, fit(x, fun.x), ls = '--')
plt.plot(x, fit(x, x0), ls = '--')

plt.scatter(1/ns[0:], -np.array(n_bs[0:]), c='b')
#plt.scatter(1/ns[0], -np.array(n_bs[0]), c = 'r')
plt.xlabel('1/N')
plt.ylabel('t')
plt.xlim([0, None])
#plt.ylim([.45, None])
#plt.plot(1/ns, n_mss)
plt.show()

In [ ]:
def fit(x, pars):
    a, b, c = pars
    return a + b * x**(9/2) + c * x**(11/2)

def cost(pars):
    return np.sum(np.square(fit(1/ns[1:], pars) - np.array(n_mss[1:])))

x0 = [.5, 0., 0.]
fun = minimize(cost, x0)
print(fun.x)

x = np.linspace(0, 0.066, 51)
plt.plot(x, fit(x, fun.x), ls = '--')
plt.plot(x, fit(x, x0), ls = '--')

plt.scatter(1/ns[1:], n_mss[1:], c='b')
plt.scatter(1/ns[0], n_mss[0], c = 'r')
plt.xlabel('1/N')
plt.ylabel(r'$n_0$')
plt.xlim([0, None])
#plt.ylim([.49, .51])
plt.show()

In [ ]:
plt.scatter(1/ns, n_mag_int)
plt.xlabel('1/N')
plt.xlim([0, None])
plt.ylim([None, None])
plt.show()